<a href="https://colab.research.google.com/github/DeepthiManthapuram/Building_LLM_Applications/blob/main/Building_Memory_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip show langchain
! pip install -qU langchain
!pip install -qU langchain-google-genai

Name: langchain
Version: 1.3.18
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google

In [17]:
from langchain.chat_models import init_chat_model
from google.colab import userdata

gemini_api_key = userdata.get('GEMINI_API_KEY')

model = init_chat_model(
model = "google_genai:gemini-3.8-flash",
api_key = gemini_api_key)

In [3]:
!pip install -qU langchain-tavily

In [5]:
#Create skill demand tool

from langchain_tavily import TavilySearch
from pprint import pprint

tavily_api_key = userdata.get('TAVILY_API_KEY')

skill_demand_tool = TavilySearch(
max_results = 5,
topic = "general",
search_depth = "advanced",
tavily_api_key = tavily_api_key)

In [6]:
#Create Job Search Tool

rapidapi_key = userdata.get('RAPIDAPI_KEY')

import requests
from langchain.tools import tool

headers = {
    "X-RapidAPI-Key": rapidapi_key,
    "X-RapidAPI-Host": "jsearch.p.rapidapi.com"
}

In [7]:
@tool
def search_jobs(skill: str, location: str) -> list:

  """Search for jobs babed on a skill and location."""

  print("Calling search jobs tool")
  print(f"Searching for {skill} jobs in {location}")

  url = "https://jsearch.p.rapidapi.com/search"

  querystring = {
  "query":f"{skill} jobs in {location}",
  "page":"1",
  "num_pages":"1",
  "country":"in",
  "employment_types": "FULLTIME,INTERN",
  "job_requirements": "under_3_years_experience,no_experience"
  }

  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()
  jobs = data.get("data", [])
  print(f"Found {len(jobs)} jobs")

  result = []
  for job in jobs:
    result.append({
    "title": job.get("job_title", ""),
    "company": job.get("employer_name", ""),
    "location": job.get("job_city", ""),
    "apply_link": job.get("job_apply_link","")
    })

  return result

In [8]:
from langchain.agents import create_agent

system_prompt = """You are a Skill-to-Career Mapping assistant.

Your job is to help students understand:
1. Skill demand
2. Career trends
3. Required skills
4. Salary information
5. Related job opportunities

Available tools:

- skill_demand_tool:
  Use this to research skill demand, career trends, salary information,
  required skills, and industry information.

- search_jobs:
  Use this to find actual job opportunities related to the requested skill.

Instructions:

1. When the user asks about a skill, use skill_demand_tool to research
   the demand and career information.

2. Use search_jobs to find related job opportunities.

3. If the user specifies a location, search jobs in that location.

4. Do not invent job listings.

5. If the job search tool returns jobs, show them clearly.

6. If no jobs are found, clearly say that no matching jobs were found.

7. Give the final answer in a clean and readable format.

Use exactly this structure:

Skill Demand
Explain the current demand for the skill in 2-4 sentences.

Key Skills Required
- Skill 1
- Skill 2
- Skill 3
- Skill 4

Salary Information
- Fresher:
- Mid-level:
- Senior:

Related Jobs
1. Job Title
   Company:
   Location:
   Apply Link:

2. Job Title
   Company:
   Location:
   Apply Link:

Career Outlook
Give a short explanation of the career opportunities.

Important:
- Do not show tool calls.
- Do not show JSON.
- Do not show Python objects.
- Do not show internal reasoning.
- Do not mention Tavily, Gemini, or tool execution.
- Give only the final readable answer."""

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
checkpointer = InMemorySaver()

In [18]:
agent = create_agent(
model = model,
tools = [skill_demand_tool, search_jobs],
system_prompt = system_prompt,
checkpointer = checkpointer,
debug = True
)

config = {
    "configurable" :{
        "thread_id":"1"
    }
}

#user_query = "What's the demand for generative AI in the industry and show me related job openings in India"
user_query = "how many jobs you found in last 1 hour"
response = agent.invoke({
  "messages": [
  {

  "role": "user",
  "content": user_query

  }
  ]
  }, config = config)

[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='f5b2dd56-e34d-49d9-8e0e-57629124f569'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "Generative AI"}'}, '__gemini_function_call_thought_signatures__': {'call_1581999': 'Es8ICswIAWkUfROwy+CIdd8U37owsyIbP53tLE+Us9sz4OVwg/VLa539dp8XZk5xFFSd6NRXD1Z4SHrltBYqNz7XPJjkMYC3V0PZ99Mocz2QLmsWZchSAPT7MmzexUx+mZBDfeksoEb9tCNE73xCT0ywvaOxRVjmNDvt8WgFprsHHfqVXmexRe4D8vg+E4i7rcsSoBxSzmHg+n+K9VzZWkx/o9hQSV8m7XUpOW7FdBaQloe4T5TyoY0vc8D3iE3aVcdgSp5ZAS8yY86yurfLS7zv+Dt2A84z+7KhwZ3eOQ3ENy5tsUcc7oelZFj4rKPsFOu3yAq1UVlTGiZTbjLpTq1aYXWE3B6DD6ehwk5zaLiUaSbmHY7j4nkfV+aUPuRQd+ZXka/3apBMLc2gwTG4o7xcsfvK1DWUbB+TfQzf4Qjddt7pU2qWQjKDcGNtkflQR9oARJJkezrzPYQqowCOlMslYFEd8HxNbLPTfoW4L0/hcj9KpJD/3tu65KCcoN9OP9eiDXbdpsQ0PBwAI/SV6LPmRz2oF6bKH0A58Zr+Qqp6J7V

In [19]:
print(response["messages"][-1].content[0]['text'])

0 jobs were found (no matching jobs were found).
